# ARC-AGI-3 — Hybrid LLM agent (Duck harness + Qwen3.8 + graph explorer)

**Solver.** Tufa Labs' open-source *Duck* harness (MIT; ARC-AGI-3 Milestone #1
winner, by Harold Bessis, Jeroen Cottaar, Isaiah Pressman, Andries Smit, Michal
Tesnar and Stefano Viel — https://github.com/Tufalabs/duck-harness): a local
LLM served by vLLM plays each game by writing Python in a sandbox that can
inspect the board (image + segmentation) and call `action(...)`.

**Changes in this notebook's bundle**
1. Model: **Qwen3.8-27B-FP8** (instead of Qwen3.6-27B-FP8).
2. Board images at **512×512** (`MULTIMODAL_UPSCALE=8`, was 4).
3. **FP8 KV cache** in vLLM.
4. **Hybrid opening**: on click-only games a graph explorer gets up to 24
   clicks on the first level before the LLM takes over (it solves some click
   puzzles in ~12–21 actions). Toggle with `HYBRID_GRAPH_OPENING=0`.
5. **Fallback**: if vLLM cannot start during the scored rerun, every game is
   played by a CPU graph-exploration agent instead of scoring 0.

**Inputs:** competition data, `driessmit1/arc3-vllm-h100-wheelhouse-v3`,
`jakobbrggen/qwen3-8-27b-fp8-hf-snapshot`. **Accelerator:** RTX Pro 6000.
Internet off.

**Save & Run All** (commit) only does a short smoke run (vLLM start + 2 games,
~30 min) to validate the stack and then writes a placeholder
`submission.parquet`. The full run happens in the competition rerun.


## 1. Mode and configuration

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

os.environ["MPLBACKEND"] = "Agg"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# RESET keeps the current level (competition semantics).
os.environ["ONLY_RESET_LEVELS"] = "true"
# Hybrid opening for click-only games (see bundle: inference/agent/graph_opening.py).
os.environ.setdefault("HYBRID_GRAPH_OPENING", "1")
os.environ.setdefault("HYBRID_GRAPH_OPENING_BUDGET", "24")
# Commit-mode smoke run: number of offline games and wall-clock cap (minutes).
COMMIT_SMOKE_GAMES = int(os.environ.get("COMMIT_SMOKE_GAMES", "2"))
COMMIT_SMOKE_MINUTES = float(os.environ.get("COMMIT_SMOKE_MINUTES", "25"))

cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
COMP_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3")
print(f"hybrid: TRUE_SUBMISSION={TRUE_SUBMISSION}")


## 2. Install the ARC runtime (offline competition wheels)

In [ ]:
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
     "--disable-pip-version-check", "--find-links", str(COMP_DIR / "arc_agi_3_wheels"), "arc-agi"],
    stdout=subprocess.DEVNULL,
)
# python-dotenv is only needed by the fallback agent framework; never fatal.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--no-index", "--no-warn-conflicts",
     "--disable-pip-version-check", "--find-links", str(COMP_DIR / "arc_agi_3_wheels"), "python-dotenv"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)


## 3. Embedded harness bundle

In [ ]:
# The TAAF source bundle (harness source + pickled solver/benchmark), embedded
# so no private dataset is needed. Extracted to a writable directory.
import base64, io, shutil, tarfile

BUNDLE_DIR = WORKING_DIR / "taaf_bundle"
shutil.rmtree(BUNDLE_DIR, ignore_errors=True)
BUNDLE_DIR.mkdir(parents=True)
BUNDLE_B64 = (
    "..."
    "9ABgGAA="
)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE_B64)), mode="r:gz") as tar:
    tar.extractall(BUNDLE_DIR)
del BUNDLE_B64
assert (BUNDLE_DIR / "taaf-kaggle-bundle.json").is_file(), "bundle marker missing"
print("hybrid: bundle extracted to", BUNDLE_DIR)


In [ ]:
# CPU graph-exploration agent, used only as the fallback when vLLM cannot start.
GRAPH_AGENT_SOURCE = "\"\"\"ARC-AGI-3 graph-exploration agent (training-free, CPU only, no internet).\n\nStrategy\n--------\nEvery level of an ARC-AGI-3 game is a deterministic-ish state machine whose\nrules are unknown.  The agent explores it systematically:\n\n1. Perception\n   * The last layer of each frame is the observed 64x64 grid.\n   * \"Ticker\" detection: step counters / energy bars change a few cells on\n     every action regardless of what the action did.  Thin, small change\n     blobs that recur on the same row/column across many transitions are\n     learned and masked, so the state hash only reflects real game state.\n   * Grid is segmented into single-colour connected components; these give\n     the candidate click targets (ACTION6), ranked into salience tiers.\n\n2. World model\n   * A directed graph: node = hash of the masked grid, edge = (candidate\n     action) -> next node / no-op / death.\n\n3. Policy\n   * Try an untested candidate action in the current node (best tier first).\n   * Otherwise BFS through known edges to the nearest node that still has\n     untested candidates and walk there (RESET is used as an edge to the\n     level start when it helps).\n   * Tiers are opened globally one at a time, so cheap / likely-useful\n     actions are exhausted everywhere before expensive ones are tried.\n   * Deadly transitions are never repeated.\n   * On level-up the graph is rebuilt for the new level; interface knowledge\n     (useless actions, ticker mask, productive click colours) carries over.\n\"\"\"\nfrom __future__ import annotations\n\nimport hashlib\nimport logging\nimport os\nimport random\nimport time\nfrom collections import deque\nfrom typing import Any, Optional\n\nimport numpy as np\nfrom arcengine import FrameData, GameAction, GameState\n\nfrom agents.agent import Agent\n\nlog = logging.getLogger()\n\n# Global wall-clock deadline shared by every game thread.  Kaggle kills the\n# notebook at 12h; stop well before that so the gateway can write its output.\n_PROCESS_START = time.time()\n_DEADLINE_SECONDS = float(os.getenv(\"ARC_AGENT_DEADLINE_SECONDS\", str(7.5 * 3600)))\n\nSIMPLE_IDS = (1, 2, 3, 4, 5, 7)\nCLICK_ID = 6\nN_TIERS = 4           # click salience tiers (simple actions live in tier 0)\nMAX_CLICKS_PER_NODE = 160\n\n\ndef _to_action(aid: int) -> GameAction:\n    return GameAction.from_id(int(aid))\n\n\ndef _grid_of(frame: FrameData) -> Optional[np.ndarray]:\n    layers = frame.frame\n    if not layers:\n        return None\n    g = np.asarray(layers[-1], dtype=np.int16)\n    if g.ndim != 2 or g.size == 0:\n        return None\n    return g\n\n\n# --------------------------------------------------------------------------\n# Connected components (4-connectivity, single colour) without scipy.\n# --------------------------------------------------------------------------\ndef _segments(grid: np.ndarray, ignore: np.ndarray) -> list[dict]:\n    h, w = grid.shape\n    seen = ignore.copy()\n    out = []\n    flat = grid.tolist()\n    seen_l = seen.tolist()\n    for y in range(h):\n        row = seen_l[y]\n        for x in range(w):\n            if row[x]:\n                continue\n            c = flat[y][x]\n            stack = [(y, x)]\n            seen_l[y][x] = True\n            cells = []\n            while stack:\n                cy, cx = stack.pop()\n                cells.append((cy, cx))\n                if cy > 0 and not seen_l[cy - 1][cx] and flat[cy - 1][cx] == c:\n                    seen_l[cy - 1][cx] = True\n                    stack.append((cy - 1, cx))\n                if cy < h - 1 and not seen_l[cy + 1][cx] and flat[cy + 1][cx] == c:\n                    seen_l[cy + 1][cx] = True\n                    stack.append((cy + 1, cx))\n                if cx > 0 and not seen_l[cy][cx - 1] and flat[cy][cx - 1] == c:\n                    seen_l[cy][cx - 1] = True\n                    stack.append((cy, cx - 1))\n                if cx < w - 1 and not seen_l[cy][cx + 1] and flat[cy][cx + 1] == c:\n                    seen_l[cy][cx + 1] = True\n                    stack.append((cy, cx + 1))\n            ys = [p[0] for p in cells]\n            xs = [p[1] for p in cells]\n            y0, y1, x0, x1 = min(ys), max(ys), min(xs), max(xs)\n            my = sum(ys) / len(ys)\n            mx = sum(xs) / len(xs)\n            # click point = member cell closest to the centroid\n            py, px = min(cells, key=lambda p: (p[0] - my) ** 2 + (p[1] - mx) ** 2)\n            out.append({\n                \"color\": c, \"area\": len(cells), \"bbox\": (y0, y1, x0, x1),\n                \"point\": (py, px),\n            })\n    return out\n\n\nclass _Ticker:\n    \"\"\"Learns cells that change on (almost) every action: counters, timers.\"\"\"\n\n    def __init__(self) -> None:\n        self.n_transitions = 0\n        self.line_hits: dict[tuple, int] = {}\n        self.line_actions: dict[tuple, set] = {}\n        self.mask: Optional[np.ndarray] = None\n        self.version = 0\n\n    def observe(self, prev: np.ndarray, cur: np.ndarray, action_id: int) -> None:\n        if prev.shape != cur.shape:\n            return\n        diff = prev != cur\n        n = int(diff.sum())\n        if n == 0:\n            return\n        self.n_transitions += 1\n        pts = np.argwhere(diff)\n        # group changed cells into small clusters (8-connectivity, cheap)\n        keys = set()\n        if n <= 400:\n            clusters = self._clusters(pts)\n            for cl in clusters:\n                ys = [p[0] for p in cl]\n                xs = [p[1] for p in cl]\n                hgt = max(ys) - min(ys) + 1\n                wid = max(xs) - min(xs) + 1\n                if len(cl) > 8:\n                    continue\n                # a counter blob sits in a fixed thin band and slides along it\n                if hgt <= 3:\n                    keys.add((\"r\", min(ys), max(ys)))\n                if wid <= 3:\n                    keys.add((\"c\", min(xs), max(xs)))\n        for k in keys:\n            self.line_hits[k] = self.line_hits.get(k, 0) + 1\n            self.line_actions.setdefault(k, set()).add(action_id)\n        self._update_mask(cur.shape)\n\n    @staticmethod\n    def _clusters(pts: np.ndarray) -> list[list[tuple[int, int]]]:\n        s = {(int(p[0]), int(p[1])) for p in pts}\n        out = []\n        while s:\n            start = s.pop()\n            stack = [start]\n            cl = [start]\n            while stack:\n                y, x = stack.pop()\n                for dy in (-1, 0, 1):\n                    for dx in (-1, 0, 1):\n                        q = (y + dy, x + dx)\n                        if q in s:\n                            s.remove(q)\n                            stack.append(q)\n                            cl.append(q)\n            out.append(cl)\n        return out\n\n    def _update_mask(self, shape: tuple[int, int]) -> None:\n        if self.n_transitions < 4:\n            return\n        lines = []\n        for k, hits in self.line_hits.items():\n            if hits >= 4 and hits >= 0.5 * self.n_transitions and (\n                len(self.line_actions[k]) >= 2 or self.n_transitions >= 12\n            ):\n                lines.append(k)\n        if not lines:\n            return\n        m = np.zeros(shape, dtype=bool)\n        for kind, a, b in lines:\n            if kind == \"r\":\n                m[a:b + 1, :] = True\n            else:\n                m[:, a:b + 1] = True\n        if self.mask is None or not np.array_equal(m, self.mask):\n            self.mask = m\n            self.version += 1\n\n\nclass _Node:\n    __slots__ = (\"key\", \"cands\", \"tiers\", \"untested\", \"edges\", \"deaths\")\n\n    def __init__(self, key: str, cands: list[tuple], tiers: list[int]) -> None:\n        self.key = key\n        self.cands = cands          # list of (action_id, (x, y) | None)\n        self.tiers = tiers          # tier per candidate\n        self.untested: list[list[int]] = [[] for _ in range(N_TIERS)]\n        for i, t in enumerate(tiers):\n            self.untested[t].append(i)\n        self.edges: dict[int, str] = {}   # cand idx -> next node key (tested, changed)\n        self.deaths: set[int] = set()\n\n    def has_untested(self, max_tier: int) -> bool:\n        for t in range(max_tier + 1):\n            if self.untested[t]:\n                return True\n        return False\n\n    def pop_untested(self, max_tier: int) -> Optional[int]:\n        for t in range(max_tier + 1):\n            if self.untested[t]:\n                return self.untested[t].pop(0)\n        return None\n\n\nclass MyAgent(Agent):\n    MAX_ACTIONS = 30000\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        kwargs[\"record\"] = False\n        if len(args) >= 5:\n            args = tuple(args[:4]) + (False,) + tuple(args[5:])\n        super().__init__(*args, **kwargs)\n        # One INFO line per action for ~110 threads would flood the log.\n        logging.getLogger().setLevel(logging.WARNING)\n        seed = int(hashlib.md5(str(self.game_id).encode()).hexdigest()[:8], 16)\n        self.rng = random.Random(seed)\n        self.ticker = _Ticker()\n        self.useless_simple: dict[int, int] = {}   # action id -> no-op count\n        self.tried_simple: dict[int, int] = {}\n        self.good_colors: dict[int, int] = {}      # click colour -> productive count\n        self._new_level(0)\n        self.prev_grid: Optional[np.ndarray] = None\n        self.prev_key: Optional[str] = None\n        self.prev_cand: Optional[int] = None\n        self.prev_action_id: Optional[int] = None\n        self.expect_reset = True\n\n    # ------------------------------------------------------------------ utils\n    def append_frame(self, frame: FrameData) -> None:\n        # The framework keeps every frame and records every action to disk.\n        # With ~110 concurrent games and tens of thousands of actions each\n        # that would exhaust RAM and disk, and nothing here needs the history,\n        # so keep only a short tail and do not record.\n        self.frames.append(frame)\n        if len(self.frames) > 8:\n            del self.frames[:-2]\n        if frame.guid:\n            self.guid = frame.guid\n\n    @property\n    def name(self) -> str:\n        return f\"{super().name}.graphx\"\n\n    def _new_level(self, level: int) -> None:\n        self.level = level\n        self.nodes: dict[str, _Node] = {}\n        self.start_key: Optional[str] = None\n        self.active_tier = 0\n        self.plan: deque = deque()\n        self.level_actions = 0\n\n    def _masked(self, grid: np.ndarray) -> np.ndarray:\n        m = self.ticker.mask\n        if m is not None and m.shape == grid.shape:\n            g = grid.copy()\n            g[m] = -1\n            return g\n        return grid\n\n    def _key(self, grid: np.ndarray) -> str:\n        g = self._masked(grid)\n        return hashlib.blake2b(g.astype(np.int8).tobytes(), digest_size=12).hexdigest() + str(g.shape)\n\n    def _time_up(self) -> bool:\n        return time.time() - _PROCESS_START > _DEADLINE_SECONDS\n\n    # ------------------------------------------------------- candidate build\n    def _build_node(self, key: str, grid: np.ndarray, avail: list[int]) -> _Node:\n        cands: list[tuple] = []\n        tiers: list[int] = []\n        simple = [a for a in SIMPLE_IDS if a in avail]\n        # demote simple actions that have been no-ops essentially always\n        for a in simple:\n            tried = self.tried_simple.get(a, 0)\n            useless = self.useless_simple.get(a, 0)\n            t = 0\n            if tried >= 12 and useless >= 0.95 * tried:\n                t = N_TIERS - 1\n            cands.append((a, None))\n            tiers.append(t)\n        if CLICK_ID in avail:\n            mask = self.ticker.mask\n            ignore = mask.copy() if mask is not None and mask.shape == grid.shape else np.zeros(grid.shape, bool)\n            segs = _segments(grid, ignore)\n            if segs:\n                vals, cnts = np.unique(grid[~ignore] if (~ignore).any() else grid, return_counts=True)\n                bg = int(vals[int(np.argmax(cnts))])\n                total = grid.size\n                color_area = dict(zip(vals.tolist(), cnts.tolist()))\n                scored = []\n                for s in segs:\n                    y0, y1, x0, x1 = s[\"bbox\"]\n                    area = s[\"area\"]\n                    c = s[\"color\"]\n                    if c == bg:\n                        tier = 2 if area <= 64 else 3\n                    elif area <= 2:\n                        tier = 1\n                    elif area <= 256:\n                        tier = 0\n                    elif area <= total * 0.2:\n                        tier = 1\n                    else:\n                        tier = 3\n                    good = self.good_colors.get(c, 0)\n                    if good > 0 and tier > 0:\n                        tier -= 1\n                    # prefer rare colours and compact shapes inside a tier\n                    rarity = color_area.get(c, 0) / total\n                    score = rarity + (0 if good else 0.05) + self.rng.random() * 0.02\n                    scored.append((tier, score, s))\n                scored.sort(key=lambda t: (t[0], t[1]))\n                for tier, _, s in scored[:MAX_CLICKS_PER_NODE]:\n                    py, px = s[\"point\"]\n                    cands.append((CLICK_ID, (int(px), int(py)), s[\"color\"]))\n                    tiers.append(min(tier, N_TIERS - 1))\n        # normalise cands to (aid, xy, color)\n        cands = [c if len(c) == 3 else (c[0], c[1], None) for c in cands]\n        return _Node(key, cands, tiers)\n\n    def _get_node(self, key: str, grid: np.ndarray, avail: list[int]) -> _Node:\n        node = self.nodes.get(key)\n        if node is None:\n            node = self._build_node(key, grid, avail)\n            self.nodes[key] = node\n        return node\n\n    # --------------------------------------------------------------- planning\n    def _bfs_to_frontier(self, src: str) -> Optional[list[int]]:\n        \"\"\"Shortest known path (list of cand idx) to a node with untested cands.\"\"\"\n        tier = self.active_tier\n        prev: dict[str, tuple[str, int]] = {src: (\"\", -1)}\n        q = deque([src])\n        while q:\n            k = q.popleft()\n            node = self.nodes.get(k)\n            if node is None:\n                continue\n            if k != src and node.has_untested(tier):\n                path = []\n                while k != src:\n                    pk, ci = prev[k]\n                    path.append(ci)\n                    k = pk\n                path.reverse()\n                return path\n            for ci, nk in node.edges.items():\n                if nk not in prev and nk in self.nodes:\n                    prev[nk] = (k, ci)\n                    q.append(nk)\n        return None\n\n    def _any_frontier(self, tier: int) -> bool:\n        return any(n.has_untested(tier) for n in self.nodes.values())\n\n    # --------------------------------------------------------------- API\n    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:\n        if latest_frame.state is GameState.WIN:\n            return True\n        return getattr(self, \"_give_up\", False) or self._time_up()\n\n    def _action(self, aid: int, xy: Optional[tuple], why: str) -> GameAction:\n        a = _to_action(aid)\n        if a.is_complex():\n            x, y = xy if xy is not None else (32, 32)\n            self._pending_data = {\"x\": int(x), \"y\": int(y)}\n        else:\n            self._pending_data = {}\n        self._pending_reason = why\n        self.prev_action_id = aid\n        return a\n\n    def do_action_request(self, action: GameAction) -> FrameData:\n        # GameAction members are process-wide singletons shared by every game\n        # thread, so never stash per-call data on them: pass it explicitly.\n        data = dict(getattr(self, \"_pending_data\", {}) or {})\n        if action.is_complex() and not data:\n            data = {\"x\": 32, \"y\": 32}\n        if not action.is_complex():\n            data = {}\n        reasoning = {\"text\": str(getattr(self, \"_pending_reason\", \"\"))}\n        self._pending_data = {}\n        raw = self.arc_env.step(action, data=data, reasoning=reasoning)\n        return self._convert_raw_frame_data(raw)\n\n    def _reset(self, why: str) -> GameAction:\n        self.prev_key = None\n        self.prev_cand = None\n        self.prev_grid = None\n        self.plan.clear()\n        self.expect_reset = True\n        self._pending_data = {}\n        self._pending_reason = why\n        self.prev_action_id = 0\n        return GameAction.RESET\n\n    def take_action(self, action: GameAction) -> Optional[FrameData]:\n        try:\n            frame = super().take_action(action)\n            self._net_failures = 0\n            return frame\n        except Exception as e:  # gateway hiccup: back off, give up if persistent\n            self._net_failures = getattr(self, \"_net_failures\", 0) + 1\n            log.warning(f\"{self.game_id}: action failed ({self._net_failures}): {e}\")\n            if self._net_failures >= 20:\n                self._give_up = True\n            time.sleep(min(5.0, 0.5 * self._net_failures))\n            return None\n\n    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:\n        try:\n            return self._choose(latest_frame)\n        except Exception as e:  # never let a bug in the policy kill the game\n            log.warning(f\"{self.game_id}: policy error {type(e).__name__}: {e}\")\n            self.plan.clear()\n            self.prev_key = None\n            self.prev_cand = None\n            self.prev_grid = None\n            if latest_frame.state in (GameState.NOT_PLAYED, GameState.GAME_OVER):\n                return self._reset(\"error -> reset\")\n            avail = [int(getattr(a, \"value\", a)) for a in (latest_frame.available_actions or [])]\n            avail = [a for a in avail if a != 0] or [1]\n            aid = self.rng.choice(avail)\n            return self._action(aid, (self.rng.randrange(64), self.rng.randrange(64)), \"error fallback\")\n\n    def _choose(self, latest_frame: FrameData) -> GameAction:\n        state = latest_frame.state\n        if state in (GameState.NOT_PLAYED,) or latest_frame.frame is None or len(latest_frame.frame) == 0:\n            return self._reset(\"start\")\n\n        grid = _grid_of(latest_frame)\n        if grid is None:\n            return self._reset(\"empty frame\")\n        avail = [int(a) if not isinstance(a, GameAction) else a.value for a in (latest_frame.available_actions or [])]\n        avail = [a for a in avail if a != 0] or [1, 2, 3, 4, 5, 6]\n\n        # ---- level change bookkeeping\n        lvl = int(latest_frame.levels_completed or 0)\n        level_up = lvl > self.level\n        if level_up:\n            if self.prev_key is not None and self.prev_cand is not None:\n                pn = self.nodes.get(self.prev_key)\n                if pn is not None:\n                    c = pn.cands[self.prev_cand]\n                    if c[0] == CLICK_ID and c[2] is not None:\n                        self.good_colors[c[2]] = self.good_colors.get(c[2], 0) + 3\n            self._new_level(lvl)\n            self.prev_key = None\n            self.prev_cand = None\n            self.prev_grid = None\n\n        # ---- update ticker mask with the last transition\n        if self.prev_grid is not None and self.prev_action_id is not None and not level_up:\n            self.ticker.observe(self.prev_grid, grid, self.prev_action_id)\n\n        key = self._key(grid)\n\n        # ---- record the outcome of the previous action\n        if self.prev_key is not None and self.prev_cand is not None:\n            pn = self.nodes.get(self.prev_key)\n            if pn is not None:\n                aid, _, color = pn.cands[self.prev_cand]\n                if state is GameState.GAME_OVER:\n                    # keep any known successor: deaths are often caused by a\n                    # hidden budget (energy / moves) rather than the edge itself\n                    pn.deaths.add(self.prev_cand)\n                else:\n                    changed = key != self.prev_key\n                    if aid != CLICK_ID:\n                        self.tried_simple[aid] = self.tried_simple.get(aid, 0) + 1\n                        if not changed:\n                            self.useless_simple[aid] = self.useless_simple.get(aid, 0) + 1\n                    elif changed and color is not None:\n                        self.good_colors[color] = self.good_colors.get(color, 0) + 1\n                    if changed:\n                        pn.edges[self.prev_cand] = key\n                    else:\n                        pn.edges.pop(self.prev_cand, None)\n\n        if state is GameState.GAME_OVER:\n            return self._reset(\"game over -> reset\")\n\n        # ticker mask changed -> old hashes are stale; rebuild graph lazily\n        if getattr(self, \"_mask_version\", 0) != self.ticker.version:\n            self._mask_version = self.ticker.version\n            self.nodes = {}\n            self.start_key = None\n            self.plan.clear()\n            key = self._key(grid)\n\n        node = self._get_node(key, grid, avail)\n        if self.start_key is None:\n            self.start_key = key\n        self.expect_reset = False\n\n        self.prev_grid = grid\n        self.prev_key = key\n        self.level_actions += 1\n\n        # ---- follow an existing plan if we are on track\n        if self.plan:\n            exp_key, ci = self.plan[0]\n            if exp_key == key:\n                self.plan.popleft()\n                self.prev_cand = ci\n                aid, xy, _ = node.cands[ci]\n                return self._action(aid, xy, \"plan\")\n            self.plan.clear()\n\n        # ---- untested action here\n        while True:\n            ci = node.pop_untested(self.active_tier)\n            if ci is not None:\n                self.prev_cand = ci\n                aid, xy, _ = node.cands[ci]\n                return self._action(aid, xy, f\"explore t{node.tiers[ci]}\")\n\n            path = self._bfs_to_frontier(key)\n            if path:\n                # store as (expected node key, cand) pairs\n                k = key\n                steps = []\n                for ci in path:\n                    steps.append((k, ci))\n                    k = self.nodes[k].edges[ci]\n                self.plan = deque(steps)\n                exp_key, ci = self.plan.popleft()\n                self.prev_cand = ci\n                aid, xy, _ = node.cands[ci]\n                return self._action(aid, xy, \"goto frontier\")\n\n            # frontier unreachable from here but reachable from level start\n            if key != self.start_key and self.start_key in self.nodes and (\n                self.nodes[self.start_key].has_untested(self.active_tier)\n                or self._bfs_to_frontier(self.start_key)\n            ):\n                return self._reset(\"frontier via start -> reset\")\n\n            if self.active_tier < N_TIERS - 1:\n                self.active_tier += 1\n                continue\n            break\n\n        # ---- everything reachable is exhausted.  The level most likely has\n        # hidden state (keys carried, energy, order-dependent switches) that the\n        # hash cannot see.  Forget the graph and explore again from here: new\n        # paths through the same screens then get tried in new contexts.\n        self.exhaust_count = getattr(self, \"exhaust_count\", 0) + 1\n        if self.exhaust_count <= 50 and len(self.nodes) > 1:\n            self.nodes = {}\n            self.start_key = None\n            self.active_tier = 0\n            self.plan.clear()\n            node = self._get_node(key, grid, avail)\n            self.start_key = key\n            ci = node.pop_untested(0)\n            if ci is not None:\n                self.prev_cand = ci\n                aid, xy, _ = node.cands[ci]\n                return self._action(aid, xy, \"re-explore\")\n        ci = self.rng.randrange(len(node.cands)) if node.cands else None\n        if ci is None:\n            return self._reset(\"no candidates\")\n        if self.rng.random() < 0.02:\n            return self._reset(\"exhausted -> reset\")\n        self.prev_cand = ci\n        aid, xy, _ = node.cands[ci]\n        return self._action(aid, xy, \"random fallback\")\n"


## 4. Locate attached datasets (vLLM wheelhouse, Qwen3.8 weights)

In [ ]:
DATASET_SOURCES = ["tharunkumar369/embedded-taaf-bundle", "driessmit1/arc3-vllm-h100-wheelhouse-v3", "jakobbrggen/qwen3-8-27b-fp8-hf-snapshot"]
KERNEL_SOURCES = []
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


def _dataset_mount_candidates(ref: str) -> list:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _first_existing(candidates):
    return next((c for c in candidates if c.exists()), None)


kaggle_input_paths = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print("hybrid: input paths =", setup_env["TAAF_KAGGLE_INPUT_PATHS"])


## 5. Start vLLM (falls back to the graph agent if this fails in the rerun)

In [ ]:
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))

# Setup = install vLLM from the wheelhouse, start the OpenAI-compatible server
# with Qwen3.8-27B-FP8, smoke-test it. If that fails in the scored rerun we fall
# back to the CPU graph agent instead of crashing (which would score 0).
LLM_READY = False
try:
    env = _command_env()
    for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
        print("hybrid: running setup command", flush=True)
        subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        env = _command_env()
        os.environ.update(env)
    for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
        if entry not in sys.path:
            sys.path.insert(0, entry)
    LLM_READY = True
    print("hybrid: LLM stack ready", flush=True)
except Exception as exc:
    log_path = WORKING_DIR / "vllm-openai-server.log"
    tail = log_path.read_text(errors="replace").splitlines()[-60:] if log_path.exists() else []
    print(f"hybrid: LLM setup FAILED: {type(exc).__name__}: {exc}", flush=True)
    print("\n".join(tail), flush=True)
    if not TRUE_SUBMISSION:
        raise  # surface the problem during Save & Run All


## 6. Load the solver and benchmark

In [ ]:
if LLM_READY:
    with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
        target = pickle.load(file)
    target.actual_run_as_submission = TRUE_SUBMISSION
    target.is_competition_rerun = TRUE_SUBMISSION
    with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
        bm = pickle.load(file)
    bm.job_dir = WORKING_DIR
    print("hybrid: benchmark loaded:", bm.label)


## 7. Play

In [ ]:
def _competition_games():
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _offline_games(env_dir: str):
    import arc_agi
    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


def _write_placeholder_submission():
    import pandas as pd

    pd.DataFrame([["1_0", "1", True, 1]], columns=["row_id", "game_id", "end_of_game", "score"]).to_parquet(
        WORKING_DIR / "submission.parquet", index=False
    )


def _run_fallback_graph_agent():
    """Play every game with the CPU graph-exploration agent via the official framework."""
    fw = WORKING_DIR / "ARC-AGI-3-Agents"
    shutil.rmtree(fw, ignore_errors=True)
    shutil.copytree(COMP_DIR / "ARC-AGI-3-Agents", fw)
    (fw / "agents" / "templates" / "my_agent.py").write_text(GRAPH_AGENT_SOURCE)
    (fw / "agents" / "__init__.py").write_text(
        "from typing import Type\n"
        "from dotenv import load_dotenv\n"
        "from .agent import Agent, Playback\n"
        "from .swarm import Swarm\n"
        "from .templates.random_agent import Random\n"
        "from .templates.my_agent import MyAgent\n\n"
        "load_dotenv()\n\n"
        "AVAILABLE_AGENTS: dict[str, Type[Agent]] = {'random': Random, 'myagent': MyAgent}\n"
    )
    (fw / ".env").write_text(
        "SCHEME=http\nHOST=gateway\nPORT=8001\nARC_API_KEY=test-key-123\n"
        "ARC_BASE_URL=http://gateway:8001/\nOPERATION_MODE=online\nENVIRONMENTS_DIR=\n"
        f"RECORDINGS_DIR={WORKING_DIR / 'server_recording'}\n"
    )
    elapsed = time.time() - NOTEBOOK_START_EPOCH
    fenv = os.environ.copy()
    fenv["ARC_AGENT_DEADLINE_SECONDS"] = str(int(max(600.0, 10.5 * 3600 - elapsed)))
    fenv["MPLBACKEND"] = "agg"
    print("hybrid: FALLBACK graph agent, deadline", fenv["ARC_AGENT_DEADLINE_SECONDS"], "s", flush=True)
    subprocess.run([sys.executable, "main.py", "--agent", "myagent"], cwd=fw, env=fenv, check=False)


import shutil

os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))
if TRUE_SUBMISSION:
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    _wait_for_gateway(os.environ["ARC_BASE_URL"])

if not LLM_READY:
    # Only reachable in the scored rerun (commit mode re-raises in setup).
    _run_fallback_graph_agent()
else:
    print((BUNDLE_DIR / "preamble.txt").read_text())
    (WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())
    if TRUE_SUBMISSION:
        bm.games = _competition_games()
        soft_end = None
    else:
        # Save & Run All: short smoke run on a few bundled public games.
        bm.games = _offline_games(str(COMP_DIR / "environment_files"))[:COMMIT_SMOKE_GAMES]
        soft_end = datetime.now() + timedelta(minutes=COMMIT_SMOKE_MINUTES)
    bm.n_passes = 1
    bm.game_weights = None
    print(f"hybrid: playing {len(bm.games)} games", flush=True)
    try:
        await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
    finally:
        for command in json.loads((BUNDLE_DIR / "teardown_commands.json").read_text()):
            subprocess.run(command, shell=True, check=False, cwd=WORKING_DIR, env=_command_env())

if not TRUE_SUBMISSION:
    # Commit runs are not scored; Kaggle still needs a submission.parquet output.
    # In the rerun the gateway writes the real one.
    _write_placeholder_submission()
    print("hybrid: wrote placeholder submission.parquet")


## 8. Diagnostics

In [ ]:
diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    from html import escape
    from IPython.display import HTML, display

    display(HTML(f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
                 'width="100%" height="900" style="border:0"></iframe>'))
else:
    print("No diagnostics.html (minimal diagnostics in the scored rerun).")
